# Specialized Production Middlewares

1. ModelCallLimitMiddleware
2. ToolCallLimitMiddleware
3. ModelFallbackMiddleware
4. HumanInTheLoopMiddleware
5. SummarizationMiddleware

### 1. ModelCallLimitMiddleware (Preventing Runaway LLM Spend)

**Goal**: Intercept every LLM call and raise an exception if the model call count exceeds max_calls.

**Why it matters**: Prevents infinite loops or unexpected high API costs.

In [1]:
import os
from dotenv import load_dotenv, find_dotenv
from langchain_core.callbacks import BaseCallbackHandler
from langchain_google_genai import ChatGoogleGenerativeAI

load_dotenv(find_dotenv())

class ModelCallLimitMiddleware(BaseCallbackHandler):
    def __init__(self, max_calls: int = 2):
        self.max_calls = max_calls
        self.call_count = 0
    
    def on_llm_start(self, serialized, prompts, **kwargs):
        self.call_count += 1
        print(f"[ModelCallLimitMiddleware] Current call count: {self.call_count}/{self.max_calls}")
        if self.call_count > self.max_calls:
            raise RuntimeError(f"STOP! ModelCallLimitMiddleware: Exceeded max allowed LLM calls ({self.max_calls}).")

limit_handler = ModelCallLimitMiddleware(max_calls=2)
llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash", callbacks=[limit_handler])

print("Call 1:")
llm.invoke("Say Hello!")
print("Call 2:")
llm.invoke("Say Goodbye!")
print("ModelCallLimitMiddleware test complete!")


Call 1:
[ModelCallLimitMiddleware] Current call count: 1/2
Call 2:
[ModelCallLimitMiddleware] Current call count: 2/2
ModelCallLimitMiddleware test complete!


### 2. ToolCallLimitMiddleware (Preventing Infinite Tool Loops)

**Goal**: Track tool execution frequency and halt execution if an agent invokes tools too many times.

**Why it matters**: Prevents agents from getting stuck in repeated tool calls.

In [1]:
from langchain_core.callbacks import BaseCallbackHandler
from langchain_core.tools import tool

class ToolCallLimitMiddleware(BaseCallbackHandler):
    def __init__(self, max_tool_calls: int = 1):
        self.max_tool_calls = max_tool_calls
        self.tool_calls = 0
    
    def on_tool_start(self, serialized, input_str, **kwargs):
        self.tool_calls += 1
        tool_name = serialized.get("name", "unknown")
        print(f"[ToolCallLimitMiddleware] Tool '{tool_name}' execution count: {self.tool_calls}/{self.max_tool_calls}")
        if self.tool_calls > self.max_tool_calls:
            raise RuntimeError(f"STOP! ToolCallLimitMiddleware: Exceeded max tool calls ({self.max_tool_calls}).")

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

tool_limit_handler = ToolCallLimitMiddleware(max_tool_calls=1)
print("ToolCallLimitMiddleware initialized successfully!")


ToolCallLimitMiddleware initialized successfully!


### 3. ModelFallbackMiddleware (Automatic Model Fallback on API Failure)

**Goal**: Automatically route requests to a secondary model if the primary model fails or encounters rate limits.

**Why it matters**: Ensures 99.9% uptime for production AI services.

In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI

# Primary model (Gemini 3.6 Flash)
primary_llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

# Fallback backup model (Gemini 2.0 Flash)
fallback_llm = ChatGoogleGenerativeAI(model="gemini-3.6-flash")

# Attach fallback model using with_fallbacks
llm_with_fallback = primary_llm.with_fallbacks([fallback_llm])

response = llm_with_fallback.invoke("Explain fallbacks in one sentence.")
print(f"Fallback Chain Response: {response.content}")


Fallback Chain Response: [{'type': 'text', 'text': 'A fallback is a backup option or alternative mechanism designed to maintain functionality when the primary choice fails or is unavailable.', 'extras': {'signature': 'ErMPCrAPARFNMg8l5sZZERVLcxYaW3Vu6fbJtcdPITTbXOcaZqEEidpOPqHcO65t2XuLvuOeQil3k2fN1Xzo0e2RLGYlS8paklzRp54wADvdlNmVEyfzZkrb5rZ69dIyFEDG/wTGqXqNI7kZB0hKDBuk5sxngh0905lU7NFO3W2XPVpzl6Bflh2rBsSSIL9Dv5XqrkHDWgFCdyKNXCIN3aXfJ2otLJfWGnxwhdW2Vep0lOsDp8vwipFBTazyst6Ht6Qr1pN9qVnncHQBIRYJNxp/VruWiMO5K/NzhuwoI1YLZNksjdN6FZgEk8p5OynwEepmwnXC3L9Ng0W8ANWVfQYxvNU/KMhQha/p6vwvKvEvwAsyC0IGOsL75a4iA2ZRfvJTCDLoTNMZWKK/73nYnZ3gaKD7eNFdKfdAyq75szuR3650defB3l1IutDycHWCaMsnwtCQAhQUa8zQIHRVcC5jybCxmmK9vtXmPzE3rmVATHAnUBnrFlyWmTRn/Q3p5hJHgSFcH5C3a3tY01XkzetJMcNlkqwxQawa2QTnICRgVU15agybJvehTZ80eOL5o9xqAicDvR+xKF4cgWB5/71Ndb1mRW/n4jSnR0tTcY9HUFQMJCriJsuc52wAAE2LIK/IDCjfnnQRNfXzs6Aoa2jMpVFcQod/re3wYMM6Q57kWUyAM3Upi18d/K+vGN3+tCRKAzF3DJXaxAhAhG09Tj1TtmY+ikIraZtTXQSLU+wHjjPiGmBSX81TqDSjzCcJv80K2uwxJZDgho

### 4. HumanInTheLoopMiddleware (Human Approval Gate)

**Goal**: Intercept tool execution to require human confirmation before running sensitive operations.

**Why it matters**: Essential for security when agents trigger database deletions, emails, or payments.

In [1]:
def human_in_the_loop_approval(tool_name: str, tool_args: dict) -> bool:
    print(f"\n⚠️ [HumanInTheLoopMiddleware] APPROVAL REQUESTED for tool '{tool_name}' with args {tool_args}")
    # Simulate human typing 'yes'
    user_input = "yes"
    if user_input.lower().strip() == "yes":
        print("✅ [HumanInTheLoopMiddleware] Human APPROVED the action!")
        return True
    else:
        print("❌ [HumanInTheLoopMiddleware] Human REJECTED the action!")
        return False

is_approved = human_in_the_loop_approval("delete_database_record", {"record_id": 101})
print(f"Execution Approved: {is_approved}")



⚠️ [HumanInTheLoopMiddleware] APPROVAL REQUESTED for tool 'delete_database_record' with args {'record_id': 101}
✅ [HumanInTheLoopMiddleware] Human APPROVED the action!
Execution Approved: True


### 5. SummarizationMiddleware (Automatic Context Trimming)

**Goal**: Trim or summarize conversation message history when context length exceeds max token limits.

**Why it matters**: Prevents LLM context overflow errors in long multi-turn chats.

In [1]:
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage, trim_messages

message_history = [
    SystemMessage(content="You are an AI assistant."),
    HumanMessage(content="My favorite sport is Cricket."),
    AIMessage(content="Cricket is a great sport!"),
    HumanMessage(content="My favorite team is India."),
    AIMessage(content="Team India has a rich cricketing history."),
    HumanMessage(content="Who is the captain?")
]

# Trim to retain last 4 messages while preserving SystemMessage
trimmed_history = trim_messages(
    message_history,
    max_tokens=4,
    token_counter=len,
    strategy="last",
    include_system=True
)

print(f"Original Messages: {len(message_history)} | Trimmed Messages: {len(trimmed_history)}\n")
for msg in trimmed_history:
    print(f"[{type(msg).__name__}]: {msg.content}")


Original Messages: 6 | Trimmed Messages: 4

[SystemMessage]: You are an AI assistant.
[HumanMessage]: My favorite team is India.
[AIMessage]: Team India has a rich cricketing history.
[HumanMessage]: Who is the captain?
